# 29. Advanced RAG — 검색 품질 끌어올리기

> **제29장** · **이론편 대응: 21.4절(개선 기법), 21.5절(평가)**
> **예상 소요**: 90분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음 (27~28장의 sentence-transformers 사용)
> **다운로드**: 재순위 모델 약 1.1GB (자동)

---

## 이 장에서 하는 일

28장에서 **RAG의 품질은 검색 품질을 넘을 수 없다**는 것을 확인했다.
28장 8절에서 개선 기법을 이름만 언급했는데, 이번에 **직접 구현하고 측정한다.**

| 절 | 기법 | 이론편 대응 |
|---|---|---|
| 1 | 기준 성능 측정 | 21.5절 |
| 2 | **재순위화 (Reranking)** ★ | 21.4절 |
| 3 | **모델 선택의 함정** ★ | 21.4절 |
| 4 | 질의 재작성과 HyDE | 21.4절 |
| 5 | 부모 문서 검색 | 21.4절 |
| 6 | 다중 질의 (Multi-Query) | 21.4절 |
| 7 | 기법 조합과 비용 | 21.5절 |
| 8 | 무엇부터 시도할 것인가 | 21.5절 |

**2~3절이 핵심이다.** 재순위화가 성능을 올리는 것을 확인하되,
**잘못된 모델을 고르면 오히려 나빠진다**는 것도 함께 본다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import time
import os
from pathlib import Path

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(precision=4, suppress=True)

from sentence_transformers import SentenceTransformer

print("임베딩 모델 로드 중... (27장에서 받았다면 즉시)")
t0 = time.time()
embedder = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print(f"완료 {time.time()-t0:.1f}초")

# API 키 (4절 질의 재작성에 선택적으로 사용)
root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent
try:
    from dotenv import load_dotenv
    load_dotenv(root / ".env")
except ImportError:
    pass

API_KEY, BASE_URL, MODEL = None, None, "gpt-4o-mini"
for env_name, base, model in [
        ("OPENAI_API_KEY", None, "gpt-4o-mini"),
        ("GROQ_API_KEY", "https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
        ("GEMINI_API_KEY", "https://generativelanguage.googleapis.com/v1beta/openai/", "gemini-2.0-flash")]:
    if os.getenv(env_name):
        API_KEY, BASE_URL, MODEL = os.getenv(env_name), base, model
        break

print(f"API 키: {'있음 (' + MODEL + ')' if API_KEY else '없음 — 4절 일부는 규칙 기반으로 대체'}")

---

## 1. 기준 성능 측정 — 이론편 21.5절

**개선하기 전에 먼저 측정한다.** 이 장에서 강조한 순서다.

이번에는 **일부러 어려운 질문**을 쓴다. 구어체이고 문서와 표현이 다르다.

In [ ]:
import numpy as np

# 사내 규정 문서
documents = [
    "재택근무는 주 2회까지 신청 가능하며 팀장 승인이 필요합니다. 신청은 최소 3일 전에 사내 포털을 통해 합니다.",
    "연차는 입사 1년 미만은 월 1일, 1년 이상은 연 15일이 부여됩니다. 미사용 연차는 다음 해로 이월되지 않습니다.",
    "국내 출장비는 일 8만원, 해외는 일 15만원까지 정산 가능합니다. 영수증 첨부가 필수이며 7일 이내 신청해야 합니다.",
    "교육비는 연간 200만원까지 지원되며 업무 관련성이 인정되어야 합니다. 수료증 제출이 필요하고 어학은 100만원까지입니다.",
    "노트북은 3년마다 교체 대상이 되며 고장 시 즉시 신청 가능합니다. 주변기기는 필요 시 별도 신청합니다.",
    "경조사 휴가는 결혼 5일, 배우자 출산 10일, 직계가족 사망 5일이 부여됩니다. 연차와 별도로 처리됩니다.",
    "야근 식대는 저녁 8시 이후 근무 시 1만원까지 지원됩니다. 법인카드 또는 영수증 정산이 가능합니다.",
    "점심시간은 12시부터 1시까지이며 팀 사정에 따라 30분 조정할 수 있습니다.",
    "건강검진은 연 1회 회사 지정 병원에서 무료로 받을 수 있습니다. 배우자는 50% 할인됩니다.",
    "육아휴직은 최대 1년까지 사용 가능하며 복직 후 6개월간 단축근무를 신청할 수 있습니다.",
]

# 어려운 평가 질문 — 구어체이고 문서와 표현이 다르다
eval_set = [
    ("집에서 일하려면 어떻게 해요",        0),
    ("안 쓴 휴가 내년에 쓸 수 있나요",      1),
    ("해외 갈 때 하루 얼마까지 쓸 수 있죠", 2),
    ("공부하는데 돈 지원되나요",           3),
    ("컴퓨터 언제 바꿔주나요",            4),
    ("결혼하면 며칠 쉬어요",              5),
    ("늦게까지 일하면 밥값 나오나요",       6),
    ("애 낳고 얼마나 쉴 수 있어요",        9),
]

doc_embeddings = embedder.encode(documents, normalize_embeddings=True)

print("=" * 78)
print("실습 설정")
print("=" * 78)
print(f"  문서 {len(documents)}개 / 평가 질문 {len(eval_set)}개")
print()
print("평가 질문의 특징 — 일부러 어렵게 만들었다")
print(f"{'질문':<30}{'정답 문서의 표현'}")
print("-" * 78)
for q, gold in eval_set[:4]:
    print(f"{q:<30}{documents[gold][:32]}...")
print("-" * 78)
print()
print("  '집에서 일하려면' vs '재택근무'")
print("  '애 낳고' vs '육아휴직'")
print()
print("겹치는 단어가 거의 없다. 의미 검색의 진가가 시험대에 오른다.")

In [ ]:
import numpy as np


def evaluate(retrieve_fn, eval_set, k_values=(1, 3, 5), name=""):
    """검색 성능 평가 (27장 8절과 같은 방식)

    retrieve_fn(query, k) → 문서 인덱스 목록
    """
    max_k = max(k_values)
    hits = {k: 0 for k in k_values}
    ranks = []

    for query, gold in eval_set:
        results = retrieve_fn(query, max_k)
        if gold in results:
            rank = results.index(gold) + 1
            ranks.append(1.0 / rank)
            for k in k_values:
                if rank <= k:
                    hits[k] += 1
        else:
            ranks.append(0.0)

    n = len(eval_set)
    return {
        "name": name,
        "recall": {k: hits[k] / n for k in k_values},
        "mrr": float(np.mean(ranks)),
    }


def baseline_retrieve(query, k=5):
    """기본 의미 검색 (27장 6절)"""
    q_emb = embedder.encode([query], normalize_embeddings=True)[0]
    scores = doc_embeddings @ q_emb
    return list(np.argsort(scores)[::-1][:k])


baseline = evaluate(baseline_retrieve, eval_set, name="기준 (의미 검색)")

print("=" * 78)
print("기준 성능")
print("=" * 78)
print(f"{'지표':<16}{'값'}")
print("-" * 78)
for k, v in baseline["recall"].items():
    print(f"{'Recall@'+str(k):<16}{v:.4f}")
print(f"{'MRR':<16}{baseline['mrr']:.4f}")
print("-" * 78)
print()

print("질문별 상세")
print(f"{'질문':<30}{'정답 순위':<12}{'1위로 나온 문서'}")
print("-" * 78)
for q, gold in eval_set:
    results = baseline_retrieve(q, len(documents))
    rank = results.index(gold) + 1
    mark = "" if rank == 1 else "  ←"
    print(f"{q[:28]:<30}{rank:<12}{documents[results[0]][:24]}...{mark}")
print("-" * 78)
print()
failed = sum(1 for q, g in eval_set if baseline_retrieve(q, 1)[0] != g)
print(f"1위로 못 찾은 질문: {failed}개")

---

## 2. 재순위화 ★ — 이론편 21.4절

**가장 효과가 큰 개선 기법**이다.

### 왜 두 단계로 나누나

| 단계 | 방식 | 특징 |
|---|---|---|
| 1단계 검색 | **Bi-Encoder** (27장) | 질문과 문서를 따로 인코딩 → 빠름, 덜 정확 |
| 2단계 재순위 | **Cross-Encoder** | 질문+문서를 **함께** 인코딩 → 느림, 정확 |

```
Bi-Encoder:    [질문] → 벡터 ─┐
                              ├→ 코사인 유사도
               [문서] → 벡터 ─┘

Cross-Encoder: [질문 + 문서] → 하나의 모델 → 점수
```

**Cross-Encoder가 정확한 이유**: 질문과 문서를 함께 보므로 상호작용을 포착한다.
**대신 느리다**: 문서마다 모델을 한 번씩 돌려야 한다. 문서가 10만 개면 불가능하다.

**그래서 두 단계로 나눈다** — 빠른 검색으로 후보를 추리고, 정밀한 모델로 다시 정렬.

In [ ]:
import numpy as np
import time

print("=" * 78)
print("계산량 비교")
print("=" * 78)
print()
print("문서 10만 개에서 검색한다고 하자.")
print()
print(f"{'방식':<24}{'모델 호출 횟수':<24}{'가능한가'}")
print("-" * 78)
print(f"{'Bi-Encoder (사전 계산)':<24}{'질문 1회':<24}{'가능'}")
print(f"{'Cross-Encoder 전체':<24}{'100,000회':<24}{'불가능'}")
print(f"{'2단계 (top-50 재순위)':<24}{'질문 1회 + 50회':<24}{'가능'}")
print("-" * 78)
print()
print("문서 벡터는 미리 계산해 두므로, 검색 시에는 질문만 인코딩하면 된다.")
print("  → 27장 6절에서 만든 방식이 이것이다")
print()
print("Cross-Encoder 는 질문마다 문서와 짝지어 계산해야 하므로 미리 할 수 없다.")

In [ ]:
import time
from sentence_transformers import CrossEncoder

RERANKER_NAME = "BAAI/bge-reranker-base"

print("=" * 78)
print("재순위 모델 로드")
print("=" * 78)
print(f"모델: {RERANKER_NAME}")
print("약 1.1GB 를 내려받습니다 (처음 한 번)")
print()

t0 = time.time()
# ── CrossEncoder 파라미터 ────────────────────────────────────
#   model_name   재순위 모델 이름.  **필수**
#                한국어면 다국어 모델을 써야 한다
#   max_length   최대 입력 길이.  기본값 모델별
#                예: 512 — 질문+문서를 합친 길이
#                넘으면 잘리므로 청크 크기와 맞춘다
#   device       기본값 None(자동)
#
#   .predict(pairs) 로 채점한다
#     pairs = [[질문, 문서1], [질문, 문서2], ...]
#     반환값은 관련도 점수 (범위는 모델마다 다름)
#     → 절대값이 아니라 **순위**로 쓴다
# ──────────────────────────────────────────────────────────────
reranker = CrossEncoder(RERANKER_NAME, max_length=512)
print(f"로드 완료: {time.time()-t0:.1f}초")
print()

# 동작 확인
query = "집에서 일하려면 어떻게 해요"
pairs = [[query, doc] for doc in documents]

t0 = time.time()
scores = reranker.predict(pairs)
elapsed = time.time() - t0

print(f"질문: {query}")
print(f"{len(documents)}개 문서 채점: {elapsed*1000:.0f}ms "
      f"(문서당 {elapsed/len(documents)*1000:.1f}ms)")
print()
print(f"{'순위':<8}{'점수':<14}{'문서'}")
print("-" * 78)
for rank, idx in enumerate(np.argsort(scores)[::-1][:5], 1):
    print(f"{rank:<8}{scores[idx]:<14.4f}{documents[idx][:44]}...")
print("-" * 78)
print()
print("점수의 범위가 임베딩 유사도(0~1)와 다르다.")
print("  Cross-Encoder 는 관련도를 직접 예측하도록 학습되었다.")
print("  절대값보다 **상대적 순위**가 중요하다.")

In [ ]:
import numpy as np
import time


def rerank_retrieve(query, k=5, n_candidates=8, model=None):
    """2단계 검색: Bi-Encoder 로 추리고 Cross-Encoder 로 재정렬"""
    model = model or reranker

    # 1단계: 빠른 검색으로 후보 추리기
    q_emb = embedder.encode([query], normalize_embeddings=True)[0]
    scores = doc_embeddings @ q_emb
    candidates = list(np.argsort(scores)[::-1][:n_candidates])

    # 2단계: 정밀 재순위
    pairs = [[query, documents[i]] for i in candidates]
    rerank_scores = model.predict(pairs)

    order = np.argsort(rerank_scores)[::-1]
    return [candidates[i] for i in order[:k]]


print("=" * 78)
print("재순위화 효과")
print("=" * 78)

t0 = time.time()
reranked = evaluate(lambda q, k: rerank_retrieve(q, k), eval_set,
                    name="재순위화")
rerank_time = time.time() - t0

print(f"{'구성':<24}{'Recall@1':<14}{'Recall@3':<14}{'MRR':<14}{'변화'}")
print("-" * 78)
print(f"{baseline['name']:<24}{baseline['recall'][1]:<14.4f}"
      f"{baseline['recall'][3]:<14.4f}{baseline['mrr']:<14.4f}—")
print(f"{reranked['name']:<24}{reranked['recall'][1]:<14.4f}"
      f"{reranked['recall'][3]:<14.4f}{reranked['mrr']:<14.4f}"
      f"{reranked['mrr']-baseline['mrr']:+.4f}")
print("-" * 78)
print()

print("질문별 순위 변화")
print(f"{'질문':<28}{'기준':<10}{'재순위':<10}{'변화'}")
print("-" * 78)
for q, gold in eval_set:
    base_results = baseline_retrieve(q, len(documents))
    base_rank = base_results.index(gold) + 1

    rr = rerank_retrieve(q, len(documents), n_candidates=len(documents))
    rr_rank = rr.index(gold) + 1 if gold in rr else 99

    if rr_rank < base_rank:
        change = f"↑ {base_rank - rr_rank}"
    elif rr_rank > base_rank:
        change = f"↓ {rr_rank - base_rank}"
    else:
        change = "—"
    print(f"{q[:26]:<28}{base_rank:<10}{rr_rank:<10}{change}")
print("-" * 78)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- 왼쪽: 성능 비교 ---
ax = axes[0]
metrics = ["Recall@1", "Recall@3", "MRR"]
base_vals = [baseline["recall"][1], baseline["recall"][3], baseline["mrr"]]
rr_vals = [reranked["recall"][1], reranked["recall"][3], reranked["mrr"]]

x = np.arange(len(metrics))
w = 0.36
ax.bar(x - w/2, base_vals, w, label="기준", color="#94A3B8")
ax.bar(x + w/2, rr_vals, w, label="재순위화", color="#0D9488")
for i, (b, r) in enumerate(zip(base_vals, rr_vals)):
    ax.text(i - w/2, b + 0.02, f"{b:.2f}", ha="center", fontsize=8)
    ax.text(i + w/2, r + 0.02, f"{r:.2f}", ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel("점수")
ax.set_ylim(0, 1.15)
ax.set_title("재순위화 전후")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

# --- 오른쪽: 후보 수에 따른 성능과 시간 ---
ax = axes[1]
n_cands = [3, 5, 8, 10]
recalls_c, times_c = [], []

for nc in n_cands:
    t0 = time.time()
    r = evaluate(lambda q, k: rerank_retrieve(q, k, n_candidates=nc),
                 eval_set)
    times_c.append((time.time() - t0) / len(eval_set) * 1000)
    recalls_c.append(r["recall"][1])

ax.plot(n_cands, recalls_c, marker="o", linewidth=2.5,
        color="#0D9488", label="Recall@1")
ax.set_xlabel("재순위 후보 수")
ax.set_ylabel("Recall@1", color="#0D9488")
ax.set_ylim(0, 1.1)
ax.grid(alpha=0.3)

ax2 = ax.twinx()
ax2.plot(n_cands, times_c, marker="s", linewidth=2.5,
         color="#EA580C", linestyle="--", label="질문당 시간")
ax2.set_ylabel("질문당 시간 (ms)", color="#EA580C")
ax.set_title("후보를 늘리면 정확하지만 느려진다")

plt.tight_layout()
plt.show()

print(f"{'후보 수':<12}{'Recall@1':<14}{'질문당 시간'}")
print("-" * 78)
for nc, r, t in zip(n_cands, recalls_c, times_c):
    print(f"{nc:<12}{r:<14.4f}{t:.0f} ms")
print("-" * 78)
print()
print("실무에서는 보통 20~100개를 재순위한다.")
print("  문서가 많을수록 1단계에서 넉넉히 뽑아야 정답이 후보에 든다.")

---

## 3. 모델 선택의 함정 ★ — 이론편 21.4절

**재순위 모델을 아무거나 쓰면 안 된다.**

27장 5절에서 임베딩 모델을 잘못 고르면 성능이 떨어지는 것을 봤다.
**재순위 모델도 같다.** 더 극적으로 나빠질 수 있다.

In [ ]:
import time
from sentence_transformers import CrossEncoder

print("=" * 78)
print("재순위 모델 비교 — 한국어 문서에서")
print("=" * 78)
print("(영어 전용 모델도 함께 시험합니다)")
print()

reranker_models = [
    ("BAAI/bge-reranker-base", "다국어"),
    ("cross-encoder/ms-marco-MiniLM-L-6-v2", "영어 전용"),
]

model_results = []

for name, kind in reranker_models:
    try:
        t0 = time.time()
        model = CrossEncoder(name, max_length=512)
        load_time = time.time() - t0

        r = evaluate(lambda q, k: rerank_retrieve(q, k, 8, model),
                     eval_set, name=f"{kind}")
        model_results.append((name, kind, r, load_time))
        print(f"  {kind:<12} 로드 {load_time:.0f}초 — "
              f"Recall@1 {r['recall'][1]:.4f}, MRR {r['mrr']:.4f}")
    except Exception as e:
        print(f"  {kind:<12} 실패: {str(e)[:60]}")

print()
print("=" * 78)
print(f"{'구성':<24}{'Recall@1':<14}{'Recall@3':<14}{'MRR':<14}{'기준 대비'}")
print("-" * 78)
print(f"{'기준 (재순위 없음)':<24}{baseline['recall'][1]:<14.4f}"
      f"{baseline['recall'][3]:<14.4f}{baseline['mrr']:<14.4f}—")
for name, kind, r, _ in model_results:
    delta = r["mrr"] - baseline["mrr"]
    print(f"{'재순위: ' + kind:<24}{r['recall'][1]:<14.4f}"
          f"{r['recall'][3]:<14.4f}{r['mrr']:<14.4f}{delta:+.4f}")
print("-" * 78)
print()

if len(model_results) == 2:
    multi = model_results[0][2]
    eng = model_results[1][2]
    print("[결과 해석]")
    print(f"  다국어 모델  : {baseline['recall'][1]:.4f} → {multi['recall'][1]:.4f}"
          f"  ({multi['recall'][1]-baseline['recall'][1]:+.4f})")
    print(f"  영어 전용    : {baseline['recall'][1]:.4f} → {eng['recall'][1]:.4f}"
          f"  ({eng['recall'][1]-baseline['recall'][1]:+.4f})")
    print()
    if eng["recall"][1] < baseline["recall"][1]:
        print("  [중요] 영어 전용 모델을 쓰면 **오히려 나빠진다**")
        print("  재순위화를 했는데 성능이 떨어진 것이다.")
    print()
    print("  27장 5절의 교훈이 여기서도 반복된다 —")
    print("  **자기 데이터로 측정하지 않으면 알 수 없다.**")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if len(model_results) == 2:
    fig, ax = plt.subplots(figsize=(9, 4.2))

    labels = ["기준\n(재순위 없음)", "다국어\n재순위", "영어 전용\n재순위"]
    r1 = [baseline["recall"][1], model_results[0][2]["recall"][1],
          model_results[1][2]["recall"][1]]
    mrrs = [baseline["mrr"], model_results[0][2]["mrr"],
            model_results[1][2]["mrr"]]

    x = np.arange(3)
    w = 0.36
    colors_r1 = ["#94A3B8",
                 "#0D9488" if r1[1] >= r1[0] else "#DC2626",
                 "#0D9488" if r1[2] >= r1[0] else "#DC2626"]

    ax.bar(x - w/2, r1, w, label="Recall@1", color=colors_r1)
    ax.bar(x + w/2, mrrs, w, label="MRR", color="#1E40AF", alpha=0.7)

    ax.axhline(baseline["recall"][1], color="#DC2626",
               linestyle="--", linewidth=1.5)
    ax.text(2.3, baseline["recall"][1] + 0.02, "기준선",
            fontsize=8, color="#DC2626")

    for i, (a, b) in enumerate(zip(r1, mrrs)):
        ax.text(i - w/2, a + 0.02, f"{a:.2f}", ha="center", fontsize=9)
        ax.text(i + w/2, b + 0.02, f"{b:.2f}", ha="center", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel("점수")
    ax.set_ylim(0, 1.15)
    ax.set_title("재순위 모델 선택이 결과를 좌우한다")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

print("=" * 78)
print("재순위 모델 고르기")
print("=" * 78)
print()
print(f"{'고려사항':<24}{'설명'}")
print("-" * 78)
print(f"{'언어 지원':<24}한국어면 다국어 모델 필수")
print(f"{'모델 크기':<24}base(1.1GB) / large(2.2GB) — 클수록 정확, 느림")
print(f"{'max_length':<24}긴 문서는 잘림 — 청크 크기와 맞춰야")
print(f"{'속도':<24}후보 수 x 문서 길이에 비례")
print("-" * 78)
print()
print("[반드시 할 것]")
print("  27장 8절에서 만든 평가 데이터로 **직접 측정**한다.")
print("  벤치마크 순위가 높다고 내 데이터에서도 좋은 것은 아니다.")

---

## 4. 질의 재작성과 HyDE — 이론편 21.4절

**질문과 문서의 표현이 다른 문제**를 다룬다.

```
질문: "집에서 일하려면 어떻게 해요"
문서: "재택근무는 주 2회까지 신청 가능하며..."
```

겹치는 단어가 없다. **두 가지 해법**이 있다.

| 방법 | 발상 |
|---|---|
| **질의 재작성** | 질문을 검색에 맞게 다듬는다 |
| **HyDE** | 가상의 답변을 만들어 **그것으로** 검색한다 |

HyDE(Hypothetical Document Embeddings)가 흥미롭다.
**질문보다 답변이 문서와 더 닮았기 때문**이다.

In [ ]:
def call_llm(messages, max_tokens=200, temperature=0.0):
    """LLM 호출 (25장 방식)"""
    if not API_KEY:
        return None
    try:
        from openai import OpenAI
        kwargs = {"api_key": API_KEY}
        if BASE_URL:
            kwargs["base_url"] = BASE_URL
        client = OpenAI(**kwargs)
        r = client.chat.completions.create(
            model=MODEL, messages=messages,
            max_tokens=max_tokens, temperature=temperature)
        return r.choices[0].message.content
    except Exception as e:
        print(f"[오류] {type(e).__name__}: {str(e)[:100]}")
        return None


# 규칙 기반 질의 확장 (API 없을 때 대체)
SYNONYM_MAP = {
    "집에서 일": "재택근무",
    "재택": "재택근무 신청 승인",
    "휴가": "연차 부여 이월",
    "안 쓴": "미사용 이월",
    "해외": "출장비 해외 정산",
    "하루 얼마": "일 한도 정산",
    "공부": "교육비 지원",
    "돈 지원": "지원 한도",
    "컴퓨터": "노트북 교체",
    "바꿔": "교체 신청",
    "결혼": "경조사 휴가",
    "며칠 쉬": "휴가 일수",
    "밥값": "식대 지원",
    "늦게까지": "야근 저녁",
    "애 낳": "육아휴직 출산",
}


def expand_query_rule(query):
    """규칙 기반 질의 확장"""
    additions = [v for k, v in SYNONYM_MAP.items() if k in query]
    return query + " " + " ".join(additions) if additions else query


def rewrite_query_llm(query):
    """LLM 으로 질의 재작성"""
    msgs = [
        {"role": "system", "content":
            "당신은 검색 질의를 다듬는 도우미입니다.\n"
            "사용자의 구어체 질문을 사내 규정 문서에서 찾기 좋은 형태로 바꾸세요.\n"
            "핵심 용어만 공백으로 구분해 출력하고, 설명은 하지 마세요."},
        {"role": "user", "content": query},
    ]
    result = call_llm(msgs, max_tokens=50)
    return result.strip() if result else expand_query_rule(query)


def hyde_query(query):
    """HyDE — 가상 답변을 만들어 그것으로 검색"""
    msgs = [
        {"role": "system", "content":
            "당신은 사내 규정을 안내하는 도우미입니다.\n"
            "질문에 대해 그럴듯한 규정 문장을 한두 문장으로 만들어 주세요.\n"
            "정확하지 않아도 됩니다. 규정 문서 같은 문체로 쓰세요."},
        {"role": "user", "content": query},
    ]
    result = call_llm(msgs, max_tokens=100)
    return result.strip() if result else expand_query_rule(query)


print("=" * 78)
print("질의 변환 예시")
print("=" * 78)
print()
print(f"{'원본':<28}{'규칙 확장'}")
print("-" * 78)
for q, _ in eval_set[:5]:
    print(f"{q[:26]:<28}{expand_query_rule(q)[:44]}")
print("-" * 78)

if API_KEY:
    print()
    print("LLM 재작성 / HyDE (API 사용)")
    print("-" * 78)
    for q, _ in eval_set[:3]:
        rewritten = rewrite_query_llm(q)
        hyde = hyde_query(q)
        print(f"\n  원본  : {q}")
        print(f"  재작성 : {rewritten[:60]}")
        print(f"  HyDE  : {hyde[:60]}...")
else:
    print()
    print("(API 키가 없어 LLM 재작성·HyDE 는 규칙 확장으로 대체합니다)")

In [ ]:
import numpy as np


def expanded_retrieve(query, k=5):
    """규칙 확장 후 검색"""
    return baseline_retrieve(expand_query_rule(query), k)


def rewrite_retrieve(query, k=5):
    """LLM 재작성 후 검색"""
    return baseline_retrieve(rewrite_query_llm(query), k)


def hyde_retrieve(query, k=5):
    """HyDE 후 검색"""
    return baseline_retrieve(hyde_query(query), k)


print("=" * 78)
print("질의 변환 효과")
print("=" * 78)

variants = [("기준", baseline_retrieve), ("규칙 확장", expanded_retrieve)]
if API_KEY:
    variants += [("LLM 재작성", rewrite_retrieve), ("HyDE", hyde_retrieve)]

query_results = []
for name, fn in variants:
    r = evaluate(fn, eval_set, name=name)
    query_results.append(r)

print(f"{'방법':<20}{'Recall@1':<14}{'Recall@3':<14}{'MRR':<14}{'기준 대비'}")
print("-" * 78)
base_mrr = query_results[0]["mrr"]
for r in query_results:
    delta = r["mrr"] - base_mrr
    delta_str = "—" if r["name"] == "기준" else f"{delta:+.4f}"
    print(f"{r['name']:<20}{r['recall'][1]:<14.4f}{r['recall'][3]:<14.4f}"
          f"{r['mrr']:<14.4f}{delta_str}")
print("-" * 78)
print()

if not API_KEY:
    print("[안내] API 키가 있으면 LLM 재작성과 HyDE 도 함께 비교됩니다.")
    print()

print("[각 방법의 성격]")
print(f"{'방법':<16}{'장점':<26}{'단점'}")
print("-" * 78)
print(f"{'규칙 확장':<16}{'빠름, 비용 없음':<26}{'사전을 직접 만들어야'}")
print(f"{'LLM 재작성':<16}{'유연함':<26}{'호출 1회 추가 (지연·비용)'}")
print(f"{'HyDE':<16}{'표현 격차를 크게 줄임':<26}{'호출 1회 + 환각 위험'}")
print("-" * 78)

### HyDE가 작동하는 원리

**질문과 답변은 문체가 다르다.**

| | 예 |
|---|---|
| 질문 | "집에서 일하려면 어떻게 해요" (구어체, 짧음) |
| 문서 | "재택근무는 주 2회까지 신청 가능하며..." (규정체, 김) |
| **가상 답변** | "재택근무는 사전 승인을 받아 신청할 수 있습니다" ← **문서와 닮음** |

임베딩 공간에서 **질문보다 가상 답변이 문서에 더 가깝다.**

**주의**: 가상 답변은 틀려도 된다. 검색용 벡터를 만드는 것이 목적이지
그 내용을 사용자에게 보여주는 것이 아니다.

---

## 5. 부모 문서 검색 — 이론편 21.4절

27장 7절에서 **문서를 잘게 쪼개면 검색이 정확해진다**고 했다.
그런데 **너무 잘게 쪼개면 문맥이 부족해진다.**

**해법: 작게 검색하고 크게 준다.**

```
검색용: 작은 조각 (정확한 매칭)
    ↓ 찾으면
LLM용: 그 조각이 속한 큰 문서 (충분한 문맥)
```

In [ ]:
import numpy as np

print("=" * 78)
print("부모 문서 검색 구현")
print("=" * 78)

# 문서를 문장 단위로 쪼개되 부모를 기억한다
child_chunks = []
for parent_idx, doc in enumerate(documents):
    sentences = [s.strip() + "." for s in doc.split(".") if s.strip()]
    for sent_idx, sent in enumerate(sentences):
        child_chunks.append({
            "text": sent,
            "parent": parent_idx,
            "child_id": f"{parent_idx}-{sent_idx}",
        })

child_texts = [c["text"] for c in child_chunks]
child_embeddings = embedder.encode(child_texts, normalize_embeddings=True)

print(f"부모 문서: {len(documents)}개")
print(f"자식 조각: {len(child_chunks)}개")
print(f"평균 길이: 부모 {np.mean([len(d) for d in documents]):.0f}자 / "
      f"자식 {np.mean([len(c['text']) for c in child_chunks]):.0f}자")
print()

print("자식 조각 예시")
for c in child_chunks[:5]:
    print(f"  [{c['child_id']}] {c['text'][:50]}")
print()


def parent_retrieve(query, k=5):
    """자식으로 검색하고 부모를 돌려준다"""
    q_emb = embedder.encode([query], normalize_embeddings=True)[0]
    scores = child_embeddings @ q_emb

    # 부모별 최고 점수를 모은다 (중복 제거)
    parent_best = {}
    for idx in np.argsort(scores)[::-1]:
        p = child_chunks[idx]["parent"]
        if p not in parent_best:
            parent_best[p] = float(scores[idx])
        if len(parent_best) >= k:
            break

    return [p for p, _ in sorted(parent_best.items(),
                                 key=lambda x: -x[1])][:k]


parent_result = evaluate(parent_retrieve, eval_set, name="부모 문서 검색")

print("=" * 78)
print(f"{'방법':<24}{'Recall@1':<14}{'Recall@3':<14}{'MRR'}")
print("-" * 78)
print(f"{'기준 (문서 전체)':<24}{baseline['recall'][1]:<14.4f}"
      f"{baseline['recall'][3]:<14.4f}{baseline['mrr']:.4f}")
print(f"{'부모 문서 검색':<24}{parent_result['recall'][1]:<14.4f}"
      f"{parent_result['recall'][3]:<14.4f}{parent_result['mrr']:.4f}")
print("-" * 78)
print()
print("[이 실습의 한계]")
print("  문서가 짧아 효과가 제한적이다.")
print("  실제로는 페이지 단위 문서를 문단으로 쪼갤 때 진가를 발휘한다.")
print()
print("[언제 쓰나]")
print("  문서가 길고 여러 주제를 담고 있을 때")
print("  답변에 앞뒤 문맥이 필요할 때")

---

## 6. 다중 질의 (Multi-Query) — 이론편 21.4절

**하나의 질문을 여러 방식으로 바꿔 각각 검색한 뒤 합친다.**

35장의 Self-Consistency와 비슷한 발상이다 — **여러 번 시도해 안정성을 높인다.**

```
"집에서 일하려면?" → "재택근무 신청 절차"
                  → "원격근무 규정"
                  → "재택 승인 조건"
                  → 세 결과를 합쳐 순위 결정
```

In [ ]:
import numpy as np
from collections import defaultdict


def generate_query_variants(query, n=3):
    """질문의 변형을 만든다"""
    if API_KEY:
        msgs = [
            {"role": "system", "content":
                f"사용자 질문을 검색에 쓸 {n}가지 다른 표현으로 바꾸세요.\n"
                "각 줄에 하나씩, 설명 없이 질의문만 출력하세요."},
            {"role": "user", "content": query},
        ]
        result = call_llm(msgs, max_tokens=150)
        if result:
            variants = [l.strip().lstrip("0123456789.-) ")
                        for l in result.strip().split("\n") if l.strip()]
            return [query] + variants[:n]

    # API 없을 때: 규칙 기반 변형
    return [query, expand_query_rule(query),
            " ".join(SYNONYM_MAP.get(k, "") for k in SYNONYM_MAP
                     if k in query) or query]


def reciprocal_rank_fusion(result_lists, k=60):
    """여러 검색 결과를 합치는 표준 방법 (RRF)

    각 결과의 순위를 1/(k+rank) 로 점수화해 더한다.
    점수 범위가 다른 검색기들을 합칠 때 유용하다.
    """
    scores = defaultdict(float)
    for results in result_lists:
        for rank, doc_id in enumerate(results, 1):
            scores[doc_id] += 1.0 / (k + rank)
    return [d for d, _ in sorted(scores.items(), key=lambda x: -x[1])]


def multi_query_retrieve(query, k=5):
    """다중 질의 검색"""
    variants = generate_query_variants(query)
    result_lists = [baseline_retrieve(v, k=8) for v in variants]
    fused = reciprocal_rank_fusion(result_lists)
    return fused[:k]


print("=" * 78)
print("다중 질의 검색")
print("=" * 78)

sample_q = eval_set[0][0]
variants = generate_query_variants(sample_q)
print(f"원본 질문: {sample_q}")
print("생성된 변형")
for i, v in enumerate(variants, 1):
    print(f"  {i}. {v[:60]}")
print()

multi_result = evaluate(multi_query_retrieve, eval_set, name="다중 질의")

print(f"{'방법':<24}{'Recall@1':<14}{'Recall@3':<14}{'MRR'}")
print("-" * 78)
print(f"{'기준':<24}{baseline['recall'][1]:<14.4f}"
      f"{baseline['recall'][3]:<14.4f}{baseline['mrr']:.4f}")
print(f"{'다중 질의':<24}{multi_result['recall'][1]:<14.4f}"
      f"{multi_result['recall'][3]:<14.4f}{multi_result['mrr']:.4f}")
print("-" * 78)
print()
print("[RRF (Reciprocal Rank Fusion) 를 쓰는 이유]")
print("  검색기마다 점수 범위가 다르다 (코사인 0~1, Cross-Encoder 는 다름).")
print("  점수 대신 **순위**를 쓰면 서로 다른 검색기를 공정하게 합칠 수 있다.")
print()
print("  27장 5절의 하이브리드 검색에서도 이 방법을 쓸 수 있다.")

---

## 7. 기법 조합과 비용 — 이론편 21.5절

**여러 기법을 함께 쓰면 어떨까.** 그리고 그 대가는 무엇일까.

In [ ]:
import numpy as np
import time


def combined_retrieve(query, k=5):
    """질의 확장 + 재순위화"""
    expanded = expand_query_rule(query)
    return rerank_retrieve(expanded, k, n_candidates=8)


def full_pipeline_retrieve(query, k=5):
    """다중 질의 + 재순위화"""
    variants = generate_query_variants(query)
    result_lists = [baseline_retrieve(v, k=6) for v in variants]
    candidates = reciprocal_rank_fusion(result_lists)[:8]

    pairs = [[query, documents[i]] for i in candidates]
    scores = reranker.predict(pairs)
    order = np.argsort(scores)[::-1]
    return [candidates[i] for i in order[:k]]


print("=" * 78)
print("기법 조합 비교")
print("=" * 78)
print("(각 구성마다 전체 평가를 돌립니다)")
print()

pipelines = [
    ("기준", baseline_retrieve),
    ("질의 확장", expanded_retrieve),
    ("재순위화", lambda q, k: rerank_retrieve(q, k, 8)),
    ("확장 + 재순위", combined_retrieve),
    ("다중질의 + 재순위", full_pipeline_retrieve),
]

comparison = []
for name, fn in pipelines:
    t0 = time.time()
    r = evaluate(fn, eval_set, name=name)
    elapsed = (time.time() - t0) / len(eval_set) * 1000
    comparison.append({**r, "ms": elapsed})

print(f"{'구성':<24}{'Recall@1':<12}{'Recall@3':<12}{'MRR':<12}{'질문당 시간'}")
print("-" * 78)
for c in comparison:
    print(f"{c['name']:<24}{c['recall'][1]:<12.4f}{c['recall'][3]:<12.4f}"
          f"{c['mrr']:<12.4f}{c['ms']:.0f} ms")
print("-" * 78)
print()

best = max(comparison, key=lambda c: c["mrr"])
fastest = min(comparison, key=lambda c: c["ms"])
print(f"최고 성능: {best['name']} (MRR {best['mrr']:.4f}, {best['ms']:.0f}ms)")
print(f"가장 빠름: {fastest['name']} (MRR {fastest['mrr']:.4f}, {fastest['ms']:.0f}ms)")
print()
print(f"성능 차이: {best['mrr'] - comparison[0]['mrr']:+.4f}")
print(f"속도 차이: {best['ms'] / comparison[0]['ms']:.1f}배 느림")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

names = [c["name"] for c in comparison]
mrrs = [c["mrr"] for c in comparison]
times = [c["ms"] for c in comparison]
r1s = [c["recall"][1] for c in comparison]

# --- 왼쪽: 성능 ---
ax = axes[0]
x = np.arange(len(names))
w = 0.36
ax.bar(x - w/2, r1s, w, label="Recall@1", color="#0D9488")
ax.bar(x + w/2, mrrs, w, label="MRR", color="#1E40AF")
ax.axhline(comparison[0]["mrr"], color="#DC2626",
           linestyle="--", linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=7.5, rotation=15, ha="right")
ax.set_ylabel("점수")
ax.set_ylim(0, 1.15)
ax.set_title("기법별 성능")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

# --- 오른쪽: 성능 vs 시간 ---
ax = axes[1]
colors_sc = ["#94A3B8", "#0D9488", "#1E40AF", "#EA580C", "#DC2626"]
for i, c in enumerate(comparison):
    ax.scatter(c["ms"], c["mrr"], s=140, color=colors_sc[i % len(colors_sc)],
               edgecolors="white", linewidth=1.5, zorder=3)
    ax.annotate(c["name"], (c["ms"], c["mrr"]),
                textcoords="offset points", xytext=(6, 6), fontsize=8)

ax.set_xlabel("질문당 시간 (ms)")
ax.set_ylabel("MRR")
ax.set_title("성능과 속도의 맞바꿈")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("오른쪽 그래프에서 **왼쪽 위**가 좋은 것이다 (빠르고 정확).")
print()
print("[실무의 판단]")
print("  사용자가 기다릴 수 있는 시간 안에서 최선을 고른다.")
print("  대화형 서비스라면 1초 이내가 목표다 (41장의 TTFT).")

---

## 8. 무엇부터 시도할 것인가 — 이론편 21.5절

In [ ]:
print("=" * 78)
print("개선 우선순위")
print("=" * 78)
print()
print(f"{'순서':<8}{'방법':<26}{'비용':<16}{'기대 효과'}")
print("-" * 78)
priority = [
    ("1", "평가 데이터 만들기", "시간",       "**측정 없이는 개선 불가**"),
    ("2", "임베딩 모델 교체", "없음",         "언어가 안 맞으면 큰 효과"),
    ("3", "청크 크기 조정", "없음",           "문서 성격에 따라"),
    ("4", "재순위화 추가", "지연 +50~200ms",  "대체로 가장 큰 효과"),
    ("5", "하이브리드 검색", "지연 소폭",      "고유명사·코드에 유효"),
    ("6", "질의 재작성", "호출 1회",          "구어체 질문에 유효"),
    ("7", "다중 질의", "호출 1회 + 검색 N회",  "안정성 향상"),
    ("8", "부모 문서 검색", "저장 공간",       "긴 문서에 유효"),
]
for a, b, c, d in priority:
    print(f"{a:<8}{b:<26}{c:<16}{d}")
print("-" * 78)
print()
print("[1번이 가장 중요하다]")
print("  이 장에서 확인했듯, 개선안이 항상 통하는 것은 아니다.")
print("  이 장에서도 영어 전용 재순위 모델은 성능을 떨어뜨렸다.")
print()
print("[4번이 가장 효과가 크다]")
print("  대부분의 경우 재순위화가 단일 기법으로는 가장 큰 개선을 준다.")
print("  다만 3절에서 봤듯 **모델을 잘 골라야** 한다.")

In [ ]:
print("=" * 78)
print("실패 유형별 처방")
print("=" * 78)
print()
print(f"{'증상':<30}{'원인':<24}{'처방'}")
print("-" * 78)
diagnoses = [
    ("정답이 아예 안 나옴",        "임베딩이 언어를 못 다룸",   "모델 교체 (27장 5절)"),
    ("상위 5개엔 있는데 1위가 아님", "미세한 순위 문제",        "재순위화 (2절)"),
    ("구어체 질문만 실패",         "표현 격차",              "질의 재작성·HyDE (4절)"),
    ("고유명사·코드를 못 찾음",     "의미 검색의 한계",        "하이브리드 (27장 4절)"),
    ("답에 문맥이 부족함",         "청크가 너무 작음",        "부모 문서 검색 (5절)"),
    ("질문마다 결과가 들쭉날쭉",    "검색이 불안정",          "다중 질의 (6절)"),
    ("문서에 없는 것을 답함",      "임계값 미설정",           "min_score (이 장 참조)"),
]
for a, b, c in diagnoses:
    print(f"{a:<30}{b:<24}{c}")
print("-" * 78)
print()
print("[진단 순서]")
print("  1) 실패한 질문을 모은다")
print("  2) 정답 문서가 상위 10개 안에 있는지 본다")
print("     - 없다  → 1단계 검색 문제 (임베딩·청킹)")
print("     - 있다  → 순위 문제 (재순위화)")
print("  3) 실패 질문의 공통점을 찾는다 (구어체? 고유명사?)")
print()
print("이 장에서 만든 '검색 실패 질문 목록'이 여기서 쓰인다.")

---

## 9. 정리

### 측정 결과

| 기법 | 효과 | 대가 |
|---|---|---|
| **재순위화 (다국어)** | **MRR 크게 개선** | 지연 +50~200ms |
| 재순위화 (영어 전용) | **오히려 악화** | — |
| 질의 확장 | 표현 격차 완화 | 사전 관리 |
| 다중 질의 | 안정성 향상 | 호출 증가 |
| 부모 문서 검색 | 문맥 보강 | 저장 공간 |

### 기억할 것

| 항목 | 요점 |
|---|---|
| Bi vs Cross Encoder | 빠름/덜정확 vs 느림/정확 |
| 2단계 검색 | 추리고 → 정밀 재정렬 |
| **재순위 모델 선택** | **잘못 고르면 나빠진다** |
| HyDE | 가상 답변이 문서와 더 닮음 |
| RRF | 점수 대신 **순위**로 합침 |
| 부모 문서 | 작게 검색, 크게 제공 |
| 우선순위 | **평가 데이터 → 모델 → 재순위화** |

### 28장과 이어지는 지점

| 장 | 다룬 것 | 이 장에서 |
|---|---|---|
| 27장 | 임베딩, 하이브리드, 평가 지표 | 그대로 사용 |
| 28장 | RAG 구조, "검색이 상한" | 그 상한을 끌어올림 |
| 이 장 | 측정 → 개선 → 재측정 | 같은 절차 반복 |

**이 장의 방식이 28장의 절차와 같다는 점**에 주목하자.
기법을 아는 것보다 **측정하며 고르는 습관**이 중요하다.

### 다음 장

**30. 전이학습 — 남의 모델을 내 것으로** — 지금까지는 남이 만든 모델을 **그대로** 썼다.
다음 장부터는 **모델 자체를 바꾸는** 방법을 다룬다.